In [11]:
import numpy as np
import math

In [ ]:
def simulate_gbm(
    S0: float,
    sigma: float,
    T: float,
    n_steps: int,
    n_paths: int,
    r: float = 0.0,
    seed:int | None = None,
):
    """
    simulate geometric brownian motion under risk-neutral measure with r = 0

    parameters
    ----------
    S0 : float
        initial stock price
    sigma : float
        annualized volatility
    T : float
        time horizon in years
    n_steps : int
        number of simulation/hedging steps
    n_paths : int
        number of monte carlo paths
    r : float
        risk-free rate
    seed : int or None
        random seed for reproducibility
    
    returns
    ----------
    paths : np.ndarray
        shape: (n_paths, n_steps + 1)
    
    """

    if S0 <= 0:
        raise ValueError("S0 must be positive")
    
    if sigma < 0:
        raise ValueError("sigma must be non-negative")
    
    if T <= 0:
        raise ValueError("T must be positive")
    
    if n_steps <= 0:
        raise ValueError("n_steps must be positive")
    
    if n_paths <= 0:
        raise ValueError("n_paths must be positive")

    dt = T / n_steps

    rng = np.random.default_rng(seed)

    # one indepenedent N(0, 1) shock for each path and timestamp
    Z = rng.standard_normal(size=(n_paths, n_steps))

    log_returns = (r -0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    log_paths = np.cumsum(log_returns, axis=1)

    paths = np.empty((n_paths, n_steps + 1), dtype=float)
    paths[:, 0] = S0
    paths[:, 1:] = S0 * np.exp(log_paths)

    return paths

In [5]:
# 30 trading days

S0 = 100.0
sigma = 0.20
T = 30 / 252
n_steps = 30
n_paths = 100_000

paths = simulate_gbm(
    S0=S0,
    sigma=sigma,
    T=T,
    n_steps=n_steps,
    n_paths=n_paths,
    seed=123
)

print(paths.shape)

(100000, 31)


In [ ]:
assert np.all(paths[:, 0] == S0)
assert np.all(paths > 0)

terminal_prices = paths[:, -1]
print(terminal_prices.mean())  # should be approx S0

100.02153466337425


In [9]:
theoretical_std = S0 * np.sqrt(np.exp(sigma**2 * T) - 1)

empirical_std = paths[:, -1].std()

print("theoretical:", theoretical_std)
print("simulated:  ", empirical_std)

theoretical: 6.908878815297882
simulated:   6.87632429753303


# simulate black scholes

In [ ]:
def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def black_scholes_call(
    S: float,
    K: float,
    sigma: float,
    T: float,
    t: float = 0.0,
    r: float = 0.0,
) -> float:
    """
    black-scholes price of a european call option

    parameters
    ----------
    S : float
        current stock price
    K : float
        strike price
    sigma : float
        annualized volatility
    T : float
        maturity time in years
    t : float
        current time in years
    r : float
        risk-free rate
    """

    if S <= 0:
        raise ValueError("S must be positive")
    
    if K <= 0:
        raise ValueError("K must be positive")

    if sigma <= 0:
        raise ValueError("sigma must be positive")

    tau = T - t

    if tau < 0:
        raise ValueError("t cannot be greater than T")

    if tau == 0:
        return max(S - K, 0.0)

    sqrt_tau = math.sqrt(tau)

    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * tau) / (sigma * sqrt_tau)
    d2 = d1 - sigma * sqrt_tau

    call_price = S * norm_cdf(d1) - K * math.exp(-r * tau) * norm_cdf(d2)

    return call_price

    

In [13]:
def black_scholes_delta(
    S: float,
    K: float,
    sigma: float,
    T: float,
    t: float = 0.0,
    r: float = 0.0,
) -> float:
    """
    black-scholes delta of a european call option
    """

    if S <= 0:
        raise ValueError("S must be positive")
    
    if K <= 0:
        raise ValueError("K must be positive")

    if sigma <= 0:
        raise ValueError("sigma must be positive")

    tau = T - t

    if tau < 0:
        raise ValueError("t cannot be greater than T")

    if tau == 0:
        if S > K:
            return 1.0
        elif S < K:
            return 0.0
        else:
            return 0.5  # convention

    sqrt_tau = math.sqrt(tau)

    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * tau) / (sigma * sqrt_tau)

    return norm_cdf(d1)

In [14]:
S = 100.0
K = 100.0
sigma = 0.20
T = 1.0
t = 0.0
r = 0.0


price = black_scholes_call(
    S=S,
    K=K,
    sigma=sigma,
    T=T,
    t=t,
    r=r
)

delta = black_scholes_delta(
    S=S,
    K=K,
    sigma=sigma,
    T=T,
    t=t,
    r=r  
)

print("call price:", price)
print("delta:     ", delta)

call price: 7.965567455405804
delta:      0.539827837277029


In [15]:
# finite difference delta test

epsilon = 1e-4

price_up = black_scholes_call(
    S=S + epsilon,
    K=K,
    sigma=sigma,
    T=T,
    t=t,
    r=r  
)

price_down = black_scholes_call(
    S=S - epsilon,
    K=K,
    sigma=sigma,
    T=T,
    t=t,
    r=r  
)

finite_difference_delta = (price_up - price_down) / (2.0 * epsilon)

print("analytical delta:       ", delta)
print("finite difference delta:", finite_difference_delta)

analytical delta:        0.539827837277029
finite difference delta: 0.5398278372581444
